# N105 · 字符串分割、解码与模式匹配

**目标：** 把前缀能否完成和模式状态作为DP状态。

**先修：** N098, N066, N104。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 词典分割；多位解码；0的合法性；通配符与正则语义差异。

**配套讲解来源：** [同名逐章意见](../../comment/105_string_segmentation_matching_dp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决四个"对字符串做结构决策"的问题，前两个是"前缀"DP，后两个是"双序列匹配"DP：

1. **单词拆分**：给字符串 s 和词典 words，问 s 能不能被切成若干个恰好都在词典里的单词。比如 `s='leetcode'`、`words=['leet','code']` 返回 `True`（切成 leet+code）；`s='catsandog'`、词典 `['cats','dog','sand','and','cat']` 返回 `False`——无论怎么切，总有一段（比如末尾的 "og"）不在词典里。
2. **解码方法计数**：数字串按规则解码：'1' 到 '26' 分别对应字母（1→A，26→Z），一串数字可以按一位或两位分组，问有多少种合法解码。比如 `'226'` 返回 `3`：`2|2|6`、`22|6`、`2|26` 三种；而 `'06'` 返回 `0`，因为开头的 0 不能单独解码，"06" 也不是 10 到 26 之间的合法两位码。
3. **通配符匹配**：模式 p 里 `?` 匹配任意单个字符，`*` 匹配**任意字符串**（包括空串），问 p 能否整串匹配 s。比如 `p='a*b'` 能匹配 `'aab'`（`*` 吃掉一个 'a'）。
4. **简化正则匹配**：模式里 `.` 匹配任意单个字符，`*` 表示**前一个元素重复任意次**（包括 0 次），同样要求整串匹配。比如 `p='c*a*b'` 能匹配 `'aab'`（`c*` 零次、`a*` 两次、`b` 一次）。

最容易踩的两个坑本章会反复强调：两种星号语义完全不同；"匹配"指覆盖**整个**字符串，在中间某段匹配成功不算数。

## 2. 基础知识：先读懂这些词

- **前缀可达性（分词）**：dp[i] 是个布尔值，表示"s 的前 i 个字符能否被完全拆成词典单词"。它是 N101 位集可达性的字符串版。
- **前缀路径计数（解码）**：dp[i] 是个数，表示"前 i 个数字有多少种合法解码"。它与分词共享"最后一段怎么切"的思考方式，只是把 True/False 换成了方案数相加。
- **0 的合法性**：解码规则里 0 不能单独成码，只能作为 10 或 20 的第二位。这就是为什么判断两位码用 `'10'<=s[i-2:i]<='26'` 这种字符串区间比较——'06' 会因为小于 '10' 被正确排除。
- **双序列匹配 DP**：wildcard 和 regex 都把状态定义成 "s 前 i 个字符与 p 前 j 个字符是否匹配"，也就是 N104 那张二维网格。
- **通配符 `*`（任意串）**：它自己就能匹配任何内容。转移分两支：星号匹配空串（模式向前走、文本不动），或星号再吞掉一个字符（文本向前走、模式不动）。
- **正则 `*`（重复前一元素）**：它必须依附于前面的原子（普通字符或 `.`）。转移也分两支：前一元素出现 0 次（模式回退两格，把"原子+星号"整体跳过），或再消耗一次（当前字符与前一原子匹配，且模式不动、文本前进一格）。
- **整串匹配 vs 子串搜索**：这些接口要求 p 覆盖 s 的全部字符，dp 的终态是右下角 `dp[m][n]`，不是"某一格为真就行"。
- **滚动数组与 old 行**：wildcard 把二维表压成一行，`old` 保存上一字符的行；它和 N104 的 lcs/edit_distance 用的是同一套手法。
- **输入合法性检查**：本章的正则实现明确规定模式里只允许普通字符、`.` 和 `*`，且 `*` 前必须有原子——开头就是 `*` 或出现 `**` 的模式直接抛错，避免转移读到无意义的位置。

## 3. 思路推导：从小例子开始

**问题 1：单词拆分（`word_break`）**

- Step 1：dp[i] = 前 i 个字符能否完全拆分。dp[0]=True（空前缀天然可拆——什么都不用切）。
- Step 2：转移看"最后一段"：如果存在某个长度 k，使得 dp[i−k] 为真且 s 的第 i−k 到 i 个字符组成的片段在词典里，那么 dp[i] 为真。
- Step 3：枚举词典里出现过的长度（而不是 1..i 全试），查一次哈希集合即可。

手算 `'leetcode'`、词典 `{'leet','code'}`：候选长度只有 {4}。dp[4]：dp[0] 为真且 s[0:4]='leet' 在词典 → 真；dp[8]：dp[4] 为真且 s[4:8]='code' 在词典 → 真。返回 True。

**问题 2：解码计数（`num_decodings`）**

- Step 1：dp[i] = 前 i 个数字的解码方式数，dp[0]=1 是地基（空串恰有一种"什么都不解"的方式）。
- Step 2：最后一码要么是一位、要么是两位。若第 i 位数字不是 '0'，它可以独立成码，贡献 dp[i−1]；若最后两位组成的字符串落在 '10'..'26' 内，可以合体成码，贡献 dp[i−2]。两支相加。
- Step 3：dp[n] 就是答案。

“运行示例”部分 用 `'226'` 演示了逐前缀的过程：

| 前缀 | 一位末码贡献 | 两位末码贡献 | 总数 |
|---|---|---|---|
| '2' | 1（2 是合法一位码） | 0（不足两位） | 1 |
| '22' | 1（接在 dp[1] 后） | 1（'22' 合法两位码，接在 dp[0] 后） | 2 |
| '226' | 2（接在 dp[2] 后） | 1（'26' 合法，接在 dp[1] 后） | 3 |

三种方案分别是 2|2|6、22|6、2|26，与 dp[3]=3 对上。

**问题 3：通配符匹配（`wildcard_match`）**

- Step 1：二维含义 dp[i][j] = s 前 i 个与 p 前 j 个是否匹配；实现时按 s 逐字符滚动一行。
- Step 2：初始化行（s 为空）：只有 p 开头连续的 `*` 能匹配空串，所以 `dp[j]=dp[j-1] and c=='*'`。
- Step 3：普通字符或 `?`：要求文本、模式的末字符对上，并继承左上角（双方各退一格）。
- Step 4：`*`：匹配空串（dp[j−1]，模式走、文本不走）或吞一个字符（old[j]，文本走、模式不走），取"或"。

手算 `s='aab'`、`p='a*b'`（每行是处理完一个 s 字符后的四格 [空, a, *, b]）：空串行为 [真,假,假,假]；第一个 'a' 后 [假,真,真,假]（'a' 对上 'a'，`*` 从"空"或"吞 a"两种方式到达）；第二个 'a' 后 [假,假,真,假]；'b' 后 [假,假,真,真]——末格为真，匹配成功。

**问题 4：简化正则（`regex_match`）**

- Step 1：dp[i][j] = s 前 i 个与 p 前 j 个是否匹配，dp[0][0]=True。
- Step 2：模式末位是 `*` 时（它和前面的原子 p[j−2] 绑定），两种选择："原子+星号"整体出现零次（dp[i][j−2]），或者当前 s 字符与该原子匹配（`p[j-2]=='.'` 或等于 s[i−1]）且此前 dp[i−1][j] 已匹配（星号再吞一个）。取"或"。
- Step 3：模式末位是普通字符或 `.` 时：要求当前字符对上并继承 dp[i−1][j−1]。

手算 `s='aab'`、`p='c*a*b'`：`c*` 出现零次跳过，`a*` 吞掉两个 'a'，`b` 对上 'b'，全程走通，返回 True。对照 `regex_match('mississippi','mis*is*p*.')` 为 False：`.` 只能兜住一个字符，模式各段拼起来盖不住整个串。

## 4. 核心代码：word_break

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def word_break(s, words):
    words = set(words)
    words.discard('')
    dp = [True] + [False] * len(s)
    lengths = {len(w) for w in words}
    for i in range(1, len(s) + 1):
        dp[i] = any((i >= k and dp[i - k] and (s[i - k:i] in words) for k in lengths))
    return dp[-1]
```

### 逐段读懂代码

**函数一：`word_break`**

```python
def word_break(s, words):
    words = set(words)
    words.discard('')
    dp = [True] + [False] * len(s)
    lengths = {len(w) for w in words}
    for i in range(1, len(s) + 1):
        dp[i] = any((i >= k and dp[i - k] and (s[i - k:i] in words) for k in lengths))
    return dp[-1]
```

- `words=set(words)`：转成哈希集合，片段查询 O(1)。
- `words.discard('')`：把空词从词典里扔掉——空词会造出长度 0 的"最后一段"，让转移原地打转。
- `lengths={len(w) for w in words}`：只收集词典中真实存在的词长，外层循环就不用从 1 到 i 瞎试。
- `any(...)`：对每个候选长度 k 检查三件事——i 够长（`i>=k`）、前缀可拆（`dp[i-k]`）、最后一段是词（切片在集合里）；任意一个长度成功，dp[i] 即为真。
- `return dp[-1]`：整串的可拆性。

**函数二：`num_decodings`**

```python
def num_decodings(s):
    if not s:
        return 0
    dp = [0] * (len(s) + 1)
    dp[0] = 1
    for i in range(1, len(s) + 1):
        if s[i - 1] != '0':
            dp[i] += dp[i - 1]
        if i >= 2 and '10' <= s[i - 2:i] <= '26':
            dp[i] += dp[i - 2]
    return dp[-1]
```

- `if not s: return 0`：空串没有可解码的内容，定义为 0（注意这与 dp[0]=1 不矛盾，dp[0] 是递推地基，不代表"空输入的答案"）。
- `dp[0]=1`：空前缀恰有一种方式，保证第一位非零时 dp[1] 能正确得到 1。
- `if s[i-1]!='0'`：当前位不是 0 才能独立成码，加上一支 dp[i−1]。这就是"0 不可独立解码"的守卫。
- `if i>=2 and '10'<=s[i-2:i]<='26'`：两位分支。用字符串区间比较而非转数字，'06'、'27' 都会被自动排除（'06'<'10'，'27'>'26'）。
- 两支是加法：方案数问题用 sum，不用 max/min。

**函数三：`wildcard_match`**

```python
def wildcard_match(s, p):
    dp = [False] * (len(p) + 1)
    dp[0] = True
    for j, c in enumerate(p, 1):
        dp[j] = dp[j - 1] and c == '*'
    for x in s:
        old = dp
        dp = [False] * (len(p) + 1)
        for j, c in enumerate(p, 1):
            dp[j] = dp[j - 1] or old[j] if c == '*' else old[j - 1] and (c == '?' or c == x)
    return dp[-1]
```

- 前两行是空串行：dp[0]=True（空对空），后面每一格只有"前面全连续是 `*`"才为真，`dp[j]=dp[j-1] and c=='*'` 一句表达。
- `old=dp; dp=[False]*(len(p)+1)`：滚动——old 是上一个文本字符的行，dp 是新行且“学习目标”部分天然为假（非空文本匹配空模式必假）。
- `*` 分支 `(dp[j-1] or old[j])`：dp[j−1] 是本行已算好的"星号匹配空串"，old[j] 是上一行的"星号已吞到这里、再吞一个 x"。
- 普通分支 `old[j-1] and (c=='?' or c==x)`：上一行左一格匹配成功，且当前模式字符与文本字符 x 对上（`?` 通吃）。
- `return dp[-1]`：整串匹配的终格。

**函数四：`regex_match`**

```python
def regex_match(s, p):
    if p.startswith('*') or '**' in p:
        raise ValueError('star requires a preceding atom')
    dp = [[False] * (len(p) + 1) for _ in range(len(s) + 1)]
    dp[0][0] = True
    for i in range(len(s) + 1):
        for j in range(1, len(p) + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 2] or (i > 0 and (p[j - 2] == '.' or p[j - 2] == s[i - 1]) and dp[i - 1][j])
            elif i > 0:
                dp[i][j] = (p[j - 1] == '.' or p[j - 1] == s[i - 1]) and dp[i - 1][j - 1]
    return dp[-1][-1]
```

- 第一行守卫：`*` 必须跟在某个原子后面，开头 `*` 或连续 `**` 都是非法模式，当场报错，保证后面 `p[j-2]` 的读取有意义。
- `dp[0][0]=True`：空串对空模式匹配。
- 外层 `for i in range(len(s)+1)` 从 0 开始：为了让 i=0 的行也算出"模式开头若干个 `x*` 匹配空串"这类格子（通配符版本是靠初始化那行做的，这里统一进循环）。
- `*` 分支 `dp[i][j-2] or (...)`：前者是"原子+星号出现零次"（模式指针回退两格）；后者要求 i>0（还有字符可吞）、当前字符与前一原子 p[j−2] 匹配、且 dp[i−1][j] 已经为真（星号已吞到 i−1，再吞一个）。
- `elif i>0`：i=0 时普通字符无法匹配任何东西，保持 False。
- `return dp[-1][-1]`：右下角是整串匹配结果。此实现保留了完整二维表，辅助空间 O(|s||p|)。

## 5. 分段实现：按顺序运行

**本章接口：** `word_break(s, words)`、`num_decodings(s)`、`wildcard_match(s, p)`、`regex_match(s, p)`

### word_break

In [1]:
def word_break(s, words):
    words = set(words)
    words.discard('')
    dp = [True] + [False] * len(s)
    lengths = {len(w) for w in words}
    for i in range(1, len(s) + 1):
        dp[i] = any((i >= k and dp[i - k] and (s[i - k:i] in words) for k in lengths))
    return dp[-1]

### num_decodings

In [2]:
def num_decodings(s):
    if not s:
        return 0
    dp = [0] * (len(s) + 1)
    dp[0] = 1
    for i in range(1, len(s) + 1):
        if s[i - 1] != '0':
            dp[i] += dp[i - 1]
        if i >= 2 and '10' <= s[i - 2:i] <= '26':
            dp[i] += dp[i - 2]
    return dp[-1]

### wildcard_match

In [3]:
def wildcard_match(s, p):
    dp = [False] * (len(p) + 1)
    dp[0] = True
    for j, c in enumerate(p, 1):
        dp[j] = dp[j - 1] and c == '*'
    for x in s:
        old = dp
        dp = [False] * (len(p) + 1)
        for j, c in enumerate(p, 1):
            dp[j] = dp[j - 1] or old[j] if c == '*' else old[j - 1] and (c == '?' or c == x)
    return dp[-1]

### regex_match

In [4]:
def regex_match(s, p):
    if p.startswith('*') or '**' in p:
        raise ValueError('star requires a preceding atom')
    dp = [[False] * (len(p) + 1) for _ in range(len(s) + 1)]
    dp[0][0] = True
    for i in range(len(s) + 1):
        for j in range(1, len(p) + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 2] or (i > 0 and (p[j - 2] == '.' or p[j - 2] == s[i - 1]) and dp[i - 1][j])
            elif i > 0:
                dp[i][j] = (p[j - 1] == '.' or p[j - 1] == s[i - 1]) and dp[i - 1][j - 1]
    return dp[-1][-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='226'; rows=[]; dp=[1]
for i in range(1,len(s)+1):
    one=dp[i-1] if s[i-1]!='0' else 0; two=dp[i-2] if i>=2 and '10'<=s[i-2:i]<='26' else 0
    dp.append(one+two); rows.append((s[:i],one,two,dp[i]))
show_table(['前缀','一位末码','两位末码','总数'],rows)

前缀,一位末码,两位末码,总数
2,1,0,1
22,1,1,2
226,2,1,3


## 7. 正确性、适用条件与复杂度

分词最后一段必须属于字典；解码最后一码长度只可能1或2，且0不可独立解码。通配符星号分为匹配空串或消耗一个字符；正则星号分为零次前一元素或再消耗一次，二者下标移动不同。

**代价：** 匹配DP O(|s||p|)时间；通配符O(|p|)辅助，正则此处O(|s||p|)辅助。分词含Python子串复制与哈希，保守O(n∑候选长度)字符工作量。解码输入为十进制数字串；简化正则只含普通字符、点和星号。

### 展开理解

**分词为什么对**：一次成功拆分的"最后一段"必然是词典里的某个词，设它长 k，那么剩下的前 i−k 个字符也必须能拆分——这正是 dp[i−k]。反过来，把"可拆的前缀"接上"一个词典词"必然得到可拆的更长前缀。所以按长度从小到大推，每个 dp[i] 都被完整地判定，不多不漏。

**解码为什么对**：任何一种解码方案的最后一码，长度只能是 1 位或 2 位。按最后一位的类型把所有方案分成两批：独立成码的方案恰好对应前 i−1 位的方案（逐个对应，一一配对）；合体成两位码的方案对应前 i−2 位的方案。两批数量相加就是 dp[i]。0 的限制由两处判断精确挡住：'0' 不可独立成码（第一支不加），'06' 这类非法两位串被区间比较排除（第二支不加），所以像 '06'、'30' 这样的串会正确地解出 0。

**通配符为什么对**：`*` 匹配的内容要么是空串，要么至少一个字符。若是空串，等价于"删掉这个星号再看"；若非空，星号吞掉的这串里最后一个字符就是 s 的当前字符，剩下的部分仍是"`*` 继续 vs s 少一格"的同型问题。两种情况覆盖了星号行为的全部可能，取"或"即可。普通字符和 `?` 的情形是平凡的一一对应。

**正则为什么对**：`x*` 这个组合的匹配量要么是 0 次，要么 ≥1 次。0 次时模式直接跳过两格；≥1 次时把第一次出现交给当前字符（必须与原子 x 对上），剩下的"再匹配若干次"仍是同一个状态 dp[i−1][j]。两种情况不重不漏地覆盖了星号的全部行为。注意它和通配符星号的下标移动方向不同：正则的"零次"是模式回退两格，通配符的"空串"是模式前进一格——因为正则星号带着前面的原子，通配符星号是光杆。

**复杂度**：两个匹配 DP 都是 |s|×|p| 格、每格常数操作，时间 O(|s||p|)。比如 100 字符串对 100 模式是 10⁴ 格，瞬间完成；通配符版滚动成一维只要 O(|p|) 额外空间，正则版此处保留整表是 O(|s||p|)。分词是 n 个前缀 × 候选长度数，加上每次切片与哈希的字符拷贝，保守估计 O(n×候选长度总数) 的字符工作量。解码是 O(n) 时间 O(n) 空间，n=10⁵ 的数字串也是十万次操作，一眨眼的事。

## 8. 单元测试：每个用例在检查什么

```python
assert word_break('leetcode', ['leet', 'code']) and (not word_break('catsandog', ['cats', 'dog', 'sand', 'and', 'cat']))
```

- 正常值 + 反例成对出现：'leetcode' 能切（leet+code），'catsandog' 不能——词典能拼出 cats+and 或 cat+sand，但总会剩下 'og' 兜不住。反例专治"贪心地每段都在词典里就返回真"的错误实现。

```python
assert num_decodings('06') == 0 and num_decodings('226') == 3
```

- '06' 是 0 合法性用例：前导 0 让两种分支都失效，答案必须是 0（如果写成数字比较 6<=26 就会错算成 1）。
- '226' 是正常值用例：三种切法对应第三节那张表，验证一位/两位两支相加都对。

```python
assert regex_match('aab', 'c*a*b') and (not regex_match('mississippi', 'mis*is*p*.'))
```

- 第一条验证 `*` 的"零次"分支（`c*` 整体跳过）与"多次"分支（`a*` 吞两个 a）。
- 第二条是 LeetCode 10 的经典反例：模式各段拼起来无法覆盖整个 'mississippi'，检验"整串匹配"而非"子串命中"的终态取法。

```python
assert regex_match('', 'a*') and wildcard_match('', '**')
```

- 空串边界：空文本要能被"可出现零次的模式"匹配。`a*` 零次恰好匹配空串；`**` 两个通配星号都匹配空串。这些格子全靠 i=0 行（或空串初始化行）正确计算。

```python
for n in range(4):
    for a in product('ab', repeat=n):
        s = ''.join(a)
        for p in product('ab?*', repeat=3):
            p = ''.join(p)
            assert wildcard_match(s, p) == wildcard_ref(s, p)
        for atoms in product(('a', 'b', '.', 'a*', 'b*', '.*'), repeat=2):
            p = ''.join(atoms)
            assert regex_match(s, p) == bool(re.fullmatch(p, s))
```

- 全量对拍：空串到长度 3 的全部 'a'/'b' 文本。通配符部分拿长度 3、字符集 'ab?*' 的全部 64 个模式，与记忆化递归参考实现 `wildcard_ref`（按星号两分支直接递归）逐一对照。正则部分拿 6 种原子拼出的全部 36 个双原子模式，与 Python 标准库 `re.fullmatch` 对照——fullmatch 正是"整串匹配"语义的权威裁判。空模式、全星号、点号组合等极端全部覆盖。

In [6]:
assert word_break('leetcode', ['leet', 'code']) and (not word_break('catsandog', ['cats', 'dog', 'sand', 'and', 'cat']))

assert num_decodings('06') == 0 and num_decodings('226') == 3

assert regex_match('aab', 'c*a*b') and (not regex_match('mississippi', 'mis*is*p*.'))

assert regex_match('', 'a*') and wildcard_match('', '**')

from itertools import product

from functools import cache

import re

@cache
def wildcard_ref(s, p):
    if not p:
        return not s
    if p[0] == '*':
        return wildcard_ref(s, p[1:]) or (bool(s) and wildcard_ref(s[1:], p))
    return bool(s) and p[0] in ('?', s[0]) and wildcard_ref(s[1:], p[1:])

for n in range(4):
    for a in product('ab', repeat=n):
        s = ''.join(a)
        for p in product('ab?*', repeat=3):
            p = ''.join(p)
            assert wildcard_match(s, p) == wildcard_ref(s, p)
        for atoms in product(('a', 'b', '.', 'a*', 'b*', '.*'), repeat=2):
            p = ''.join(atoms)
            assert regex_match(s, p) == bool(re.fullmatch(p, s))

print('N105: 所有本章断言通过')

N105: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 比较*在通配符和简化正则中的不同含义。

**练习 2：** 给每类匹配写独立契约。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（两种星号的差异）**：提示——各找一个"同输入不同结果"的对子。比如文本 'cb'：通配符模式 `'a*b'` 与正则模式 `'a*b'`，一个星号当任意串、一个只重复 'a'，结果一真一假，先手算再跑代码验证。再从下标移动角度总结：正则星号的"零次"让模式指针回退两格（因为要连原子一起丢），通配符星号的"空串"让模式指针前进一格；把两段转移代码并排抄下来逐字比对。边界上想清楚：正则里 `.` 后跟 `*`（即 `.*`）为什么几乎等于通配符的 `*`，差在哪。
- **练习 2（给每类匹配写独立契约）**：提示——契约要说清四件事：输入的合法字符集、模式的合法形态（比如正则 `*` 前必须有原子）、空输入的定义（num_decodings 空串是 0，regex 空模式只匹配空串）、返回值语义（布尔还是计数、整串还是子串）。给 word_break、num_decodings、wildcard_match、regex_match 各写三行：合法输入、边界约定、一个手算示例；再各自设计一条"专门会让错误实现翻车"的断言（如 '06'、'**'、'a' 对 'ab'）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def word_break(s, words):
    words = set(words)
    words.discard('')
    dp = [True] + [False] * len(s)
    lengths = {len(w) for w in words}
    for i in range(1, len(s) + 1):
        dp[i] = any((i >= k and dp[i - k] and (s[i - k:i] in words) for k in lengths))
    return dp[-1]

def num_decodings(s):
    if not s:
        return 0
    dp = [0] * (len(s) + 1)
    dp[0] = 1
    for i in range(1, len(s) + 1):
        if s[i - 1] != '0':
            dp[i] += dp[i - 1]
        if i >= 2 and '10' <= s[i - 2:i] <= '26':
            dp[i] += dp[i - 2]
    return dp[-1]

def wildcard_match(s, p):
    dp = [False] * (len(p) + 1)
    dp[0] = True
    for j, c in enumerate(p, 1):
        dp[j] = dp[j - 1] and c == '*'
    for x in s:
        old = dp
        dp = [False] * (len(p) + 1)
        for j, c in enumerate(p, 1):
            dp[j] = dp[j - 1] or old[j] if c == '*' else old[j - 1] and (c == '?' or c == x)
    return dp[-1]

def regex_match(s, p):
    if p.startswith('*') or '**' in p:
        raise ValueError('star requires a preceding atom')
    dp = [[False] * (len(p) + 1) for _ in range(len(s) + 1)]
    dp[0][0] = True
    for i in range(len(s) + 1):
        for j in range(1, len(p) + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 2] or (i > 0 and (p[j - 2] == '.' or p[j - 2] == s[i - 1]) and dp[i - 1][j])
            elif i > 0:
                dp[i][j] = (p[j - 1] == '.' or p[j - 1] == s[i - 1]) and dp[i - 1][j - 1]
    return dp[-1][-1]

# 示例与回归测试
assert word_break('leetcode', ['leet', 'code']) and (not word_break('catsandog', ['cats', 'dog', 'sand', 'and', 'cat']))

assert num_decodings('06') == 0 and num_decodings('226') == 3

assert regex_match('aab', 'c*a*b') and (not regex_match('mississippi', 'mis*is*p*.'))

assert regex_match('', 'a*') and wildcard_match('', '**')

from itertools import product

from functools import cache

import re

@cache
def wildcard_ref(s, p):
    if not p:
        return not s
    if p[0] == '*':
        return wildcard_ref(s, p[1:]) or (bool(s) and wildcard_ref(s[1:], p))
    return bool(s) and p[0] in ('?', s[0]) and wildcard_ref(s[1:], p[1:])

for n in range(4):
    for a in product('ab', repeat=n):
        s = ''.join(a)
        for p in product('ab?*', repeat=3):
            p = ''.join(p)
            assert wildcard_match(s, p) == wildcard_ref(s, p)
        for atoms in product(('a', 'b', '.', 'a*', 'b*', '.*'), repeat=2):
            p = ''.join(atoms)
            assert regex_match(s, p) == bool(re.fullmatch(p, s))

print('N105: 所有本章断言通过')

N105: 所有本章断言通过


## 11. 代表题与迁移

- **139. Word Break（单词拆分）**：前缀可达性 DP 的原型，练"看最后一段"的转移与词典长度的枚举优化，对应 `word_break`。
- **91. Decode Ways（解码方法）**：前缀路径计数，练一位/两位两支相加以及 0 的合法性处理，对应 `num_decodings`。
- **44. Wildcard Matching（通配符匹配）**：双序列匹配 DP + 一维滚动，练 `?`/`*`（任意串语义）的两分支转移，对应 `wildcard_match`。
- **10. Regular Expression Matching（正则匹配）**：双序列匹配 DP，练 `.`/`*`（重复前一元素语义）的回退两格转移，与 44 放在一起正好体会两种星号的差别，对应 `regex_match`。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。